In [ ]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/starter.py

--2026-07-18 09:05:50--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/starter.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 907 [text/plain]
Saving to: ‘starter.py’

starter.py          100%[===================>]     907  --.-KB/s    in 0s      

2026-07-18 09:05:50 (36.2 MB/s) - ‘starter.py’ saved [907/907]



In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

# Q1. First trace
Wrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of RAGBase that wraps rag(), search(), and llm() each in their own span.

Run this query:

How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?

- 1
- 3
- 5
- 7

In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xc43f109dbbe0e09c9667739c96371967",
        "span_id": "0x8eb53139c5e23c55",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc184e4511678dfb0",
    "start_time": "2026-07-18T09:50:41.265936Z",
    "end_time": "2026-07-18T09:50:41.267840Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2d0c7826-db9c-4537-8da7-4dab160f1787",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xc43f109dbbe0e09c9667739c96371967",
        "span_id": "0xcde6900093c8cac0",
        "trace_state": "[]"
    },
    "kind": "SpanKind

# Q2. Capturing metrics as span attributes
Spans are not just timing markers - you can attach any information you want to them with set_attribute. We already use spans to record how long each step takes. Now we'll add the metrics we care about: tokens and cost.

Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span:

span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
And since we know both input and output tokens, we can also compute the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?
- 700
- 7000
- 70000
- 700000

# Q3. Span timing
Each span automatically records its duration. Look at the console output from Q1 and find the durations for the search span and the llm span.

For a typical query, roughly how long does the LLM call take?

- Under 100ms
- 100-500ms
- 500-2000ms
- Over 2000ms

# Q4. Saving traces to SQLite
Right now the spans are printed to the terminal and then gone. We don't save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there. But we need to create a custom exporter. Instead of printing the spans, it will save them to the database.

OTel calls the exporter through the same span processor we already use, we just swap the destination.

Now we will create a custom exporter that saves each finished span to a SQLite database. The exporter extends SpanExporter. It has the following methods:

export method that receives a list of ReadableSpan objects
shutdown and force_flush methods

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from starter import SQLiteSpanExporter

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))
trace.set_tracer_provider(provider)

import starter

query = "How does the agentic loop keep calling the model until it stops?"
answer = starter.rag.rag(query)

print(answer)

('The agentic loop keeps calling the model until it stops by wrapping the interaction in a `while` loop. Here\'s how it works:\n\n1.  **Initial Call:** The conversation, including instructions and the user\'s question, is sent to the LLM.\n2.  **Process Response:** The LLM\'s response is processed.\n3.  **Check for Function Calls:** A flag (e.g., `has_function_calls`) is set to `True` if the model\'s response contains any function calls.\n    *   If function calls are present, they are executed (e.g., calling the `search` tool). The output of these function calls is then appended to the message history.\n    *   If no function calls are present, it means the model has generated a final answer message.\n4.  **Loop Continuation:** The loop continues as long as the `has_function_calls` flag is `True`. This means if the model requested and received tool results, the loop will make another API call, sending the *entire* updated message history back to the model.\n5.  **Stopping Condition:**

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
pd.read_sql_query("SELECT * FROM spans", conn)

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784373459865209217,1784373459873481043,None,None,None
1,llm,1784373459878513480,1784373466405180809,None,None,None
2,rag,1784373459865094202,1784373466407820247,None,None,None
3,search,1784373658465477554,1784373658470738788,None,None,None
4,llm,1784373658476943434,1784373662728100540,None,None,None
5,rag,1784373658465416199,1784373662732450129,None,None,None
6,search,1784373670522942973,1784373670527967876,None,None,None
7,llm,1784373670544371325,1784373674455389265,None,None,None
8,rag,1784373670522882239,1784373674459509817,None,None,None
9,search,1784373700181210712,1784373700199908506,None,None,None


In [2]:
pd.read_sql_query("""
SELECT
    name,
    SUM(end_time - start_time) AS total_duration
FROM spans
WHERE name != 'rag'
GROUP BY name
ORDER BY total_duration DESC
""", conn)

,name,total_duration
0,llm,24227082630
1,search,39180000
